# Session 2: Deployment and Scaling (Instructor Notebook)

In this session students learn how to take LLM applications from prototype to production. We cover API design with FastAPI patterns, caching strategies, monitoring and observability, model routing for cost optimization, and cost tracking.

> **Instructor Note:** This notebook contains all 5 demos (identical to student) plus fully solved versions of all 4 tasks with approach explanations.

## Learning Objectives

By the end of this session, students will be able to:

1. **Design** a FastAPI service for LLM inference
2. **Implement** semantic caching to reduce API costs and latency
3. **Add** monitoring and structured logging for LLM applications
4. **Build** a model router that selects models based on complexity
5. **Track** token usage and costs across multiple models

In [ ]:
# ============================================================
# Imports and Setup
# ============================================================

import openai
import json
import time
import hashlib
import logging
from datetime import datetime
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import numpy as np

print("All imports successful!")

---
## Demos (Follow Along)

### Demo 1: Designing an LLM API Service

Production LLM applications need a clean API layer. We design a service class that wraps the LLM with request validation, error handling, and structured responses — the same patterns you would use in a FastAPI endpoint.

In [ ]:
# Demo 1 - LLM API Service Design

@dataclass
class LLMRequest:
    """Validated request for the LLM service."""
    prompt: str
    model: str = "gpt-4o-mini"
    max_tokens: int = 500
    temperature: float = 0.7
    request_id: str = ""
    
    def __post_init__(self):
        if not self.request_id:
            self.request_id = hashlib.md5(f"{self.prompt}{time.time()}".encode()).hexdigest()[:8]
        if self.max_tokens < 1 or self.max_tokens > 4096:
            raise ValueError("max_tokens must be between 1 and 4096")
        if self.temperature < 0 or self.temperature > 2:
            raise ValueError("temperature must be between 0 and 2")

@dataclass
class LLMResponse:
    """Structured response from the LLM service."""
    content: str
    model: str
    request_id: str
    tokens_used: int
    latency_ms: float
    status: str = "success"
    error: Optional[str] = None

class LLMService:
    """Production LLM service with validation and error handling."""
    
    def __init__(self):
        self.client = openai.OpenAI()
    
    def invoke(self, request: LLMRequest) -> LLMResponse:
        start = time.time()
        try:
            response = self.client.chat.completions.create(
                model=request.model,
                messages=[{"role": "user", "content": request.prompt}],
                max_tokens=request.max_tokens,
                temperature=request.temperature
            )
            latency = (time.time() - start) * 1000
            return LLMResponse(
                content=response.choices[0].message.content,
                model=request.model,
                request_id=request.request_id,
                tokens_used=response.usage.total_tokens,
                latency_ms=round(latency, 2)
            )
        except Exception as e:
            latency = (time.time() - start) * 1000
            return LLMResponse(
                content="", model=request.model,
                request_id=request.request_id,
                tokens_used=0, latency_ms=round(latency, 2),
                status="error", error=str(e)
            )

# Test the service
service = LLMService()
req = LLMRequest(prompt="What is RAG in 2 sentences?", max_tokens=100, temperature=0)
resp = service.invoke(req)

print(f"Request ID: {resp.request_id}")
print(f"Status: {resp.status}")
print(f"Model: {resp.model}")
print(f"Tokens: {resp.tokens_used}")
print(f"Latency: {resp.latency_ms}ms")
print(f"Response: {resp.content}")

### Demo 2: Semantic Caching for LLM Responses

LLM calls are expensive and slow. Semantic caching stores previous responses and returns them for similar (not just identical) queries — reducing costs by 30-70% in typical production workloads.

In [ ]:
# Demo 2 - Semantic Caching

class SemanticCache:
    """Cache that matches by semantic similarity, not exact string match."""
    
    def __init__(self, similarity_threshold=0.92):
        self.client = openai.OpenAI()
        self.cache = []
        self.threshold = similarity_threshold
        self.hits = 0
        self.misses = 0
    
    def _embed(self, text):
        return self.client.embeddings.create(
            model="text-embedding-3-small", input=[text]
        ).data[0].embedding
    
    def _cosine_sim(self, a, b):
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    
    def get(self, query):
        query_emb = self._embed(query)
        best_sim, best_response = 0, None
        for cached_emb, cached_query, cached_response in self.cache:
            sim = self._cosine_sim(query_emb, cached_emb)
            if sim > best_sim:
                best_sim = sim
                best_response = (cached_query, cached_response)
        if best_sim >= self.threshold:
            self.hits += 1
            return {"hit": True, "similarity": best_sim,
                    "cached_query": best_response[0], "response": best_response[1]}
        self.misses += 1
        return {"hit": False, "similarity": best_sim}
    
    def put(self, query, response):
        emb = self._embed(query)
        self.cache.append((emb, query, response))

# Test
cache = SemanticCache(similarity_threshold=0.90)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

q1 = "What is retrieval augmented generation?"
result = cache.get(q1)
print(f"Query: {q1}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")

answer = llm.invoke([HumanMessage(content=q1)]).content
cache.put(q1, answer)
print(f"Cached response: {answer[:100]}...")

q2 = "Explain RAG to me"
result = cache.get(q2)
print(f"\nQuery: {q2}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")
if result['hit']:
    print(f"Matched to: {result['cached_query']}")

q3 = "How does gravity work?"
result = cache.get(q3)
print(f"\nQuery: {q3}")
print(f"Cache: {result['hit']} (sim={result['similarity']:.4f})")
print(f"\nStats: {cache.hits} hits, {cache.misses} misses")

### Demo 3: Monitoring and Structured Logging

In production, you need to know what your LLM is doing. Structured logging captures every request, response, token count, latency, and error in a queryable format.

In [ ]:
# Demo 3 - Monitoring and Structured Logging

class LLMMonitor:
    def __init__(self):
        self.logs = []
        self.metrics = defaultdict(list)
    
    def log_request(self, request_id, model, prompt, response, tokens, latency_ms, status="success"):
        entry = {
            "timestamp": datetime.now().isoformat(),
            "request_id": request_id, "model": model,
            "prompt_preview": prompt[:100],
            "response_preview": response[:100] if response else "",
            "tokens": tokens, "latency_ms": latency_ms, "status": status
        }
        self.logs.append(entry)
        self.metrics["latency"].append(latency_ms)
        self.metrics["tokens"].append(tokens)
        self.metrics["models"].append(model)
        return entry
    
    def get_summary(self):
        if not self.logs:
            return "No requests logged."
        latencies = self.metrics["latency"]
        tokens = self.metrics["tokens"]
        models = self.metrics["models"]
        return {
            "total_requests": len(self.logs),
            "avg_latency_ms": round(np.mean(latencies), 2),
            "p95_latency_ms": round(np.percentile(latencies, 95), 2),
            "total_tokens": sum(tokens),
            "avg_tokens": round(np.mean(tokens), 2),
            "model_distribution": {m: models.count(m) for m in set(models)},
            "error_rate": sum(1 for l in self.logs if l["status"] != "success") / len(self.logs)
        }

# Test
monitor = LLMMonitor()
client = openai.OpenAI()

for prompt in ["What is RAG?", "Explain vector databases.", "How does chunking work?", "What is LangChain?", "Compare FAISS and ChromaDB."]:
    start = time.time()
    resp = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], max_tokens=80)
    latency = (time.time() - start) * 1000
    entry = monitor.log_request(
        request_id=hashlib.md5(prompt.encode()).hexdigest()[:8], model="gpt-4o-mini",
        prompt=prompt, response=resp.choices[0].message.content,
        tokens=resp.usage.total_tokens, latency_ms=round(latency, 2)
    )
    print(f"[{entry['request_id']}] {entry['prompt_preview'][:40]:40s} | {entry['tokens']}tok | {entry['latency_ms']}ms")

print("\n--- Summary ---")
for k, v in monitor.get_summary().items():
    print(f"  {k}: {v}")

### Demo 4: Model Routing for Cost Optimization

Not every query needs GPT-4. A model router classifies query complexity and sends simple queries to cheaper/faster models, reserving expensive models for complex tasks.

In [ ]:
# Demo 4 - Model Routing

class ModelRouter:
    MODEL_TIERS = {
        "simple": {"model": "gpt-4o-mini", "cost_per_1k": 0.00015},
        "medium": {"model": "gpt-4o-mini", "cost_per_1k": 0.00015},
        "complex": {"model": "gpt-4o", "cost_per_1k": 0.005}
    }
    
    def __init__(self):
        self.classifier = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    def classify_complexity(self, query):
        response = self.classifier.invoke([
            SystemMessage(content="""Classify this query's complexity. Return ONLY one word:
- simple: factual lookups, definitions, short answers
- medium: explanations, comparisons, moderate reasoning
- complex: multi-step reasoning, code generation, analysis"""),
            HumanMessage(content=query)
        ])
        complexity = response.content.strip().lower()
        return complexity if complexity in self.MODEL_TIERS else "medium"
    
    def route(self, query):
        complexity = self.classify_complexity(query)
        tier = self.MODEL_TIERS[complexity]
        return {"complexity": complexity, "model": tier["model"], "cost_per_1k": tier["cost_per_1k"]}

# Test
router = ModelRouter()
for q in ["What does RAG stand for?", "Compare ChromaDB vs FAISS vs Pinecone.", "Write a RAG pipeline with reranking and caching.", "What is an embedding?", "Design a multi-agent code review system."]:
    result = router.route(q)
    print(f"[{result['complexity']:7s}] -> {result['model']:12s} (${result['cost_per_1k']}/1k) | {q[:60]}")

### Demo 5: Token Usage and Cost Tracking

Tracking costs is critical for budget management. This demo builds a cost tracker that monitors token usage per model, calculates running costs, and provides spending alerts.

In [ ]:
# Demo 5 - Token Usage and Cost Tracking

class CostTracker:
    PRICING = {
        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
        "gpt-4o": {"input": 0.005, "output": 0.015},
        "text-embedding-3-small": {"input": 0.00002, "output": 0}
    }
    
    def __init__(self, budget_limit=1.0):
        self.budget_limit = budget_limit
        self.records = []
    
    def record(self, model, input_tokens, output_tokens, request_id=""):
        pricing = self.PRICING.get(model, {"input": 0.001, "output": 0.002})
        cost = (input_tokens * pricing["input"] + output_tokens * pricing["output"]) / 1000
        entry = {"timestamp": datetime.now().isoformat(), "model": model,
                 "input_tokens": input_tokens, "output_tokens": output_tokens,
                 "cost": cost, "request_id": request_id}
        self.records.append(entry)
        total = sum(r["cost"] for r in self.records)
        if total > self.budget_limit * 0.8:
            print(f"  WARNING: Budget at {total/self.budget_limit*100:.1f}%")
        return entry
    
    def get_report(self):
        total_cost = sum(r["cost"] for r in self.records)
        by_model = defaultdict(lambda: {"requests": 0, "input_tokens": 0, "output_tokens": 0, "cost": 0})
        for r in self.records:
            by_model[r["model"]]["requests"] += 1
            by_model[r["model"]]["input_tokens"] += r["input_tokens"]
            by_model[r["model"]]["output_tokens"] += r["output_tokens"]
            by_model[r["model"]]["cost"] += r["cost"]
        return {"total_cost": total_cost, "total_requests": len(self.records),
                "budget_remaining": self.budget_limit - total_cost, "by_model": dict(by_model)}

# Test
tracker = CostTracker(budget_limit=0.05)
client = openai.OpenAI()

for model, prompt in [("gpt-4o-mini", "What is RAG?"), ("gpt-4o-mini", "List three vector databases."), ("gpt-4o", "Design a production RAG architecture."), ("gpt-4o-mini", "What is cosine similarity?")]:
    resp = client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}], max_tokens=100)
    entry = tracker.record(model, resp.usage.prompt_tokens, resp.usage.completion_tokens)
    print(f"[{model:12s}] {prompt[:40]:40s} | ${entry['cost']:.6f}")

print("\n--- Cost Report ---")
report = tracker.get_report()
print(f"Total cost: ${report['total_cost']:.6f}")
print(f"Budget remaining: ${report['budget_remaining']:.6f}")
for model, stats in report['by_model'].items():
    print(f"  {model}: {stats['requests']} reqs, ${stats['cost']:.6f}")

---
## Tasks (Solved)

### Task 1: Build a Rate-Limited LLM Service

Create an LLM service wrapper that enforces rate limits — both requests-per-minute and tokens-per-minute.

> **Approach:** We maintain a list of (timestamp, token_count) entries. Before each request, we prune entries older than 60 seconds, then check if adding a new request would exceed either the RPM or TPM limit. If so, we return a rate-limit error instead of calling the API.

In [ ]:
# Task 1 — SOLUTION: Rate-Limited LLM Service

class RateLimitedLLM:
    def __init__(self, rpm_limit=10, tpm_limit=5000):
        self.rpm_limit = rpm_limit
        self.tpm_limit = tpm_limit
        self.client = openai.OpenAI()
        self.request_log = []  # List of (timestamp, token_count)
    
    def _prune_old_entries(self):
        """Remove entries older than 60 seconds."""
        cutoff = time.time() - 60
        self.request_log = [(ts, tok) for ts, tok in self.request_log if ts > cutoff]
    
    def _check_limits(self):
        """Check if we're within rate limits."""
        self._prune_old_entries()
        current_rpm = len(self.request_log)
        current_tpm = sum(tok for _, tok in self.request_log)
        
        if current_rpm >= self.rpm_limit:
            return False, f"RPM limit reached ({current_rpm}/{self.rpm_limit})"
        if current_tpm >= self.tpm_limit:
            return False, f"TPM limit reached ({current_tpm}/{self.tpm_limit})"
        return True, "OK"
    
    def invoke(self, prompt, max_tokens=200):
        """Make a rate-limited LLM call."""
        allowed, reason = self._check_limits()
        if not allowed:
            return {"status": "rate_limited", "reason": reason, "content": None}
        
        try:
            response = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens
            )
            tokens = response.usage.total_tokens
            self.request_log.append((time.time(), tokens))
            return {
                "status": "success",
                "content": response.choices[0].message.content,
                "tokens": tokens
            }
        except Exception as e:
            return {"status": "error", "reason": str(e), "content": None}


# Test with a low limit to trigger rate limiting
rl_llm = RateLimitedLLM(rpm_limit=3, tpm_limit=1000)

for i in range(5):
    result = rl_llm.invoke(f"Question {i}: What is RAG? Answer in one sentence.")
    if result['status'] == 'success':
        print(f"Request {i}: {result['status']} ({result['tokens']} tokens) - {result['content'][:60]}...")
    else:
        print(f"Request {i}: {result['status']} - {result.get('reason', '')}")

### Task 2: Implement a Response Streaming Simulator

Build a streaming response handler that processes tokens as they arrive, tracks partial results, and provides real-time latency metrics.

> **Approach:** We use OpenAI's streaming API (`stream=True`) and iterate over chunks. We track the timestamp of the first chunk (TTFT), count chunks as they arrive, and compute tokens-per-second from the total time and chunk count.

In [ ]:
# Task 2 — SOLUTION: Response Streaming Handler

class StreamingHandler:
    def __init__(self):
        self.client = openai.OpenAI()
    
    def stream(self, prompt, model="gpt-4o-mini", max_tokens=200):
        """Stream a response and track metrics."""
        start_time = time.time()
        first_token_time = None
        chunks = []
        token_count = 0
        
        response = self.client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            stream=True
        )
        
        for chunk in response:
            if chunk.choices and chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                if first_token_time is None:
                    first_token_time = time.time()
                chunks.append(content)
                token_count += 1
        
        end_time = time.time()
        total_time = end_time - start_time
        ttft = (first_token_time - start_time) * 1000 if first_token_time else 0
        tps = token_count / total_time if total_time > 0 else 0
        
        return {
            "content": "".join(chunks),
            "token_count": token_count,
            "ttft_ms": round(ttft, 2),
            "total_time_ms": round(total_time * 1000, 2),
            "tokens_per_sec": round(tps, 1)
        }


# Test
handler = StreamingHandler()

for prompt in ["Explain RAG in 3 sentences.", "What are embeddings?"]:
    result = handler.stream(prompt)
    print(f"Q: {prompt}")
    print(f"  TTFT: {result['ttft_ms']:.0f}ms | Total: {result['total_time_ms']:.0f}ms | Tokens/sec: {result['tokens_per_sec']}")
    print(f"  Response: {result['content'][:100]}...\n")

### Task 3: Build a Multi-Tier Caching System

Build a caching system with two tiers: exact-match (fast, hash-based) and semantic-match (slower, embedding-based).

> **Approach:** We use a dict keyed by prompt hash for O(1) exact matches, and an embedding-based list for semantic matches. The query method checks exact first, then semantic, then falls through to the LLM. Each tier tracks its own hit rate.

In [ ]:
# Task 3 — SOLUTION: Multi-Tier Caching System

class TieredCache:
    def __init__(self, semantic_threshold=0.92):
        self.exact_cache = {}  # hash -> response
        self.semantic_cache = []  # (embedding, query, response)
        self.threshold = semantic_threshold
        self.embed_client = openai.OpenAI()
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.stats = {"exact_hits": 0, "semantic_hits": 0, "misses": 0}
    
    def _hash(self, text):
        return hashlib.md5(text.strip().lower().encode()).hexdigest()
    
    def _embed(self, text):
        return self.embed_client.embeddings.create(
            model="text-embedding-3-small", input=[text]
        ).data[0].embedding
    
    def _cosine_sim(self, a, b):
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    
    def query(self, prompt):
        # Tier 1: Exact match
        key = self._hash(prompt)
        if key in self.exact_cache:
            self.stats["exact_hits"] += 1
            return {"content": self.exact_cache[key], "tier": "exact", "cached": True}
        
        # Tier 2: Semantic match
        query_emb = self._embed(prompt)
        best_sim, best_response = 0, None
        for cached_emb, cached_query, cached_response in self.semantic_cache:
            sim = self._cosine_sim(query_emb, cached_emb)
            if sim > best_sim:
                best_sim = sim
                best_response = cached_response
        
        if best_sim >= self.threshold:
            self.stats["semantic_hits"] += 1
            # Also add to exact cache for future exact matches
            self.exact_cache[key] = best_response
            return {"content": best_response, "tier": "semantic", "similarity": best_sim, "cached": True}
        
        # Tier 3: LLM call
        self.stats["misses"] += 1
        response = self.llm.invoke([HumanMessage(content=prompt)]).content
        
        # Store in both caches
        self.exact_cache[key] = response
        self.semantic_cache.append((query_emb, prompt, response))
        
        return {"content": response, "tier": "llm", "cached": False}


# Test
cache = TieredCache(semantic_threshold=0.90)

test_queries = [
    "What is RAG?",          # LLM call (first time)
    "What is RAG?",          # Exact match
    "Explain RAG to me",     # Semantic match
    "How does gravity work?",# LLM call (different topic)
    "How does gravity work?",# Exact match
]

for q in test_queries:
    result = cache.query(q)
    tier_info = f"tier={result['tier']}"
    if 'similarity' in result:
        tier_info += f" sim={result['similarity']:.3f}"
    print(f"[{tier_info:25s}] {q[:40]:40s} | {result['content'][:50]}...")

print(f"\nStats: {cache.stats}")

### Task 4: Create a Full Production LLM Gateway

Combine everything into a production gateway: model routing, caching, rate limiting, monitoring, and cost tracking.

> **Approach:** We compose the components from previous demos/tasks into a single gateway class. The pipeline is: classify complexity -> check cache -> check rate limit -> call LLM -> cache response -> log & track cost. The dashboard method aggregates metrics from all components.

In [ ]:
# Task 4 — SOLUTION: Full Production LLM Gateway

class LLMGateway:
    MODEL_TIERS = {
        "simple": "gpt-4o-mini",
        "medium": "gpt-4o-mini",
        "complex": "gpt-4o"
    }
    
    def __init__(self, rpm_limit=20, tpm_limit=10000, budget=1.0):
        self.client = openai.OpenAI()
        self.classifier = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        
        # Cache
        self.cache = {}  # hash -> (response, model)
        
        # Rate limiter
        self.rpm_limit = rpm_limit
        self.tpm_limit = tpm_limit
        self.request_log = []
        
        # Monitoring
        self.logs = []
        
        # Cost tracking
        self.cost_tracker = CostTracker(budget_limit=budget)
    
    def _hash(self, text):
        return hashlib.md5(text.strip().lower().encode()).hexdigest()
    
    def _classify(self, query):
        response = self.classifier.invoke([
            SystemMessage(content="Classify complexity. Return ONLY: simple, medium, or complex"),
            HumanMessage(content=query)
        ])
        c = response.content.strip().lower()
        return c if c in self.MODEL_TIERS else "medium"
    
    def _check_rate_limit(self):
        cutoff = time.time() - 60
        self.request_log = [(ts, tok) for ts, tok in self.request_log if ts > cutoff]
        if len(self.request_log) >= self.rpm_limit:
            return False, "RPM limit exceeded"
        if sum(tok for _, tok in self.request_log) >= self.tpm_limit:
            return False, "TPM limit exceeded"
        return True, "OK"
    
    def query(self, prompt):
        start = time.time()
        
        # Step 1: Route
        complexity = self._classify(prompt)
        model = self.MODEL_TIERS[complexity]
        
        # Step 2: Check cache
        key = self._hash(prompt)
        if key in self.cache:
            cached_resp, cached_model = self.cache[key]
            latency = (time.time() - start) * 1000
            return {"content": cached_resp, "model": cached_model,
                    "cached": True, "complexity": complexity,
                    "latency_ms": round(latency, 2), "cost": 0.0}
        
        # Step 3: Check rate limit
        allowed, reason = self._check_rate_limit()
        if not allowed:
            return {"content": None, "model": model, "cached": False,
                    "complexity": complexity, "status": "rate_limited", "reason": reason}
        
        # Step 4: Call LLM
        response = self.client.chat.completions.create(
            model=model, messages=[{"role": "user", "content": prompt}], max_tokens=200
        )
        content = response.choices[0].message.content
        tokens = response.usage.total_tokens
        latency = (time.time() - start) * 1000
        
        # Step 5: Cache, log, track
        self.cache[key] = (content, model)
        self.request_log.append((time.time(), tokens))
        cost_entry = self.cost_tracker.record(model, response.usage.prompt_tokens, response.usage.completion_tokens)
        self.logs.append({"timestamp": datetime.now().isoformat(), "prompt": prompt[:80],
                          "model": model, "tokens": tokens, "latency_ms": round(latency, 2)})
        
        return {"content": content, "model": model, "cached": False,
                "complexity": complexity, "latency_ms": round(latency, 2),
                "cost": cost_entry["cost"], "tokens": tokens}
    
    def get_dashboard(self):
        cost_report = self.cost_tracker.get_report()
        return {
            "total_requests": len(self.logs),
            "cache_size": len(self.cache),
            "total_cost": f"${cost_report['total_cost']:.6f}",
            "budget_remaining": f"${cost_report['budget_remaining']:.6f}",
            "models_used": cost_report['by_model']
        }


# Test
gateway = LLMGateway(budget=0.10)

test_prompts = [
    "What is RAG?",
    "What is RAG?",  # cached
    "Compare ChromaDB and FAISS.",
    "Write a Python function to compute cosine similarity.",
    "What is an embedding?"
]

for q in test_prompts:
    result = gateway.query(q)
    cached = "CACHED" if result.get('cached') else result.get('complexity', 'n/a')
    cost = result.get('cost', 0)
    print(f"[{cached:8s}] {result['model']:12s} ${cost:.6f} | {q[:50]}")

print("\n--- Dashboard ---")
for k, v in gateway.get_dashboard().items():
    print(f"  {k}: {v}")

---
## Summary

In this session students learned the operational skills for production LLM deployment:

1. **API Design** -- How to structure LLM services with validation and error handling.
2. **Semantic Caching** -- How to reduce costs by caching similar (not just identical) queries.
3. **Monitoring** -- How to track latency, tokens, errors, and model usage.
4. **Model Routing** -- How to route queries to cheaper models when possible.
5. **Cost Tracking** -- How to monitor spending and set budget alerts.

**Up Next:** After lunch, students choose their capstone track and build a complete system.